In [1]:
from sedona.spark import SedonaContext
from pyspark.sql import SparkSession

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1794925904908_0004,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
## Enable spatial indexing for fast route-crosses-polygon joins ----
spark.conf.set("sedona.global.index", "true")
spark.conf.set("sedona.join.gridtype", "kdbtree")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
## LOAD DATA SET FROM SILVER
SILVER_TRIPS_PATH   = "s3://workshop-sedona-datalake/silver/routes_simplified.geoparquet/"
SILVER_PLACES_PATH = "s3://workshop-sedona-datalake/silver/places.parquet/"

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
## Load TRIPS
df_trips_sl = spark.read.format("geoparquet").load(SILVER_TRIPS_PATH)
df_trips_sl.createOrReplaceTempView("routes_simplified")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
## Load OSM places (polygons) ----
df_places_sl = spark.read.parquet(SILVER_PLACES_PATH)
df_places_sl.createOrReplaceTempView("places")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
## ---- VIZ DATA

In [6]:
df_places_sl.groupBy("place_type").count().show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------+-----+
|       place_type|count|
+-----------------+-----+
|            scrub|    3|
|recreation_ground|   35|
|        reservoir|    8|
|       industrail|    1|
|       allotments|  291|
|           apiary|    1|
|          orchard|   67|
|          railway|   49|
|             park|    1|
|          harbour|    4|
|        flowerbed|   18|
|       healthcare|    1|
|           quarry|    5|
|          garages|   46|
|             wall|    1|
|            depot|    1|
|              ree|    1|
|            grass| 5496|
|    plant_nursery|    7|
|      residential| 4350|
+-----------------+-----+
only showing top 20 rows

In [7]:
df_trips_sl.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+--------+------------------+--------------------+------+--------------------+
|            TRIP_ID| TAXI_ID|     length_meters|                geom|points|           geom_bbox|
+-------------------+--------+------------------+--------------------+------+--------------------+
|1403190600620000571|20000571|   42317.505294946|LINESTRING (-8.48...|   162|{-8.654724, 40.89...|
|1403222177620000903|20000903| 3381.052178675152|LINESTRING (-8.64...|    23|{-8.675451, 41.14...|
|1403235920620000007|20000007| 766.0601247749518|LINESTRING (-8.60...|     8|{-8.604369, 41.15...|
|1403235910620000451|20000451| 1049.484550942543|LINESTRING (-8.60...|     8|{-8.604657, 41.15...|
|1403237750620000184|20000184| 1197.384005156106|LINESTRING (-8.61...|     7|{-8.611092, 41.14...|
|1403233081620000503|20000503|1760.8142220620477|LINESTRING (-8.68...|     6|{-8.687916, 41.16...|
|1403235404620000140|20000140| 3108.707714355438|LINESTRING (-8.60...|    12|{-8.602569, 41.15...|
|140323564

In [ ]:
## ---- END VIZ DATA

In [ ]:
# ---- ANALISYS 1

In [8]:
# ---- THE PASS-THROUGH JOIN: route crosses place polygon ----
pass_through = spark.sql("""
SELECT
    p.place_name,
    p.place_type,
    COUNT(DISTINCT r.TRIP_ID) AS routes_passing_through
FROM routes_simplified r
JOIN places p
    ON ST_Intersects(r.geom, p.geom)   -- route physically crosses the place
GROUP BY p.place_name, p.place_type
ORDER BY routes_passing_through DESC
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
print("🏆 Most Passed-Through Places in Porto:")
pass_through.show(truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

? Most Passed-Through Places in Porto:
+--------------------+------------+----------------------+
|place_name          |place_type  |routes_passing_through|
+--------------------+------------+----------------------+
|railway - 21759     |railway     |3786                  |
|railway - 21761     |railway     |2090                  |
|railway - 21766     |railway     |2060                  |
|construction - 21763|construction|1798                  |
|construction - 21764|construction|1429                  |
|construction - 21765|construction|1213                  |
|military - 1217     |military    |942                   |
|construction - 13276|construction|873                   |
|construction - 13761|construction|814                   |
|industrial - 68     |industrial  |763                   |
|grass - 11383       |grass       |722                   |
|residential - 8235  |residential |713                   |
|grass - 11382       |grass       |663                   |
|grass - 11387   

In [11]:
# ---- Flexibility: filter by place_type ----
spark.sql("""
SELECT 
    p.place_name, COUNT(DISTINCT r.TRIP_ID) AS routes
FROM routes_simplified r
JOIN 
    places p ON ST_Intersects(r.geom, p.geom)
WHERE 
    p.place_type = 'recreation_ground'           -- most-crossed parks
GROUP BY p.place_name
ORDER BY routes DESC
LIMIT 10
""").show(truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------------+------+
|place_name               |routes|
+-------------------------+------+
|recreation_ground - 3933 |46    |
|recreation_ground - 6312 |22    |
|recreation_ground - 15678|5     |
|recreation_ground - 3877 |1     |
|recreation_ground - 7437 |1     |
|recreation_ground - 616  |1     |
|recreation_ground - 21990|1     |
|recreation_ground - 8196 |1     |
+-------------------------+------+

In [ ]:
# ---- END ANALISYS 1

In [12]:
# ---- Heatmap: explode route points → bin into grid ----
# This shows the TRUE density of where cars actually drove
heatmap = spark.sql("""
WITH route_pts AS (
    SELECT
        TRIP_ID,
        ST_PointN(geom, pos) AS pt
    FROM routes_simplified
    LATERAL VIEW posexplode(
        sequence(1, ST_NPoints(geom))              -- each point is a row line
    ) p AS idx, pos
)
SELECT
    ROUND(ST_X(pt) / 0.002) * 0.002 AS grid_lon,   -- ~200m cells
    ROUND(ST_Y(pt) / 0.002) * 0.002 AS grid_lat,
    COUNT(*)                        AS density,
    COUNT(DISTINCT TRIP_ID)         AS unique_trips
FROM route_pts
GROUP BY 1, 2
ORDER BY density DESC
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
heatmap.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+--------+-------+------------+
|          grid_lon|grid_lat|density|unique_trips|
+------------------+--------+-------+------------+
|             -8.61|  41.146|   3028|        1384|
|            -8.586|  41.148|   2965|         880|
|            -8.608|  41.148|   2609|        1516|
|            -8.612|  41.146|   2402|        1297|
|            -8.626|  41.152|   2360|        1486|
|            -8.614|  41.142|   2335|         947|
|            -8.628|  41.158|   2271|         930|
|            -8.612|  41.148|   2154|        1449|
|             -8.63|  41.158|   2126|         824|
|-8.620000000000001|  41.148|   1975|        1158|
+------------------+--------+-------+------------+
only showing top 10 rows

In [ ]:
#"snap to grid" technique - ˜200m
#density = the total number of GPS points that fell inside a given ~200m grid cell.
#Alternative: H3 or geohash

In [ ]:
#CTE (route_pts)	Explodes each route LineString into its individual GPS points (one row per point)
#ST_X / ST_Y	Extracts longitude/latitude from each point
#ROUND(x/0.002)*0.002	Snaps each coordinate to a ~200m grid cell

In [ ]:
# ---- END Heatmap

In [ ]:
# ---- Merge routes

In [25]:
# ---- Merge routes per place into one representative geometry ----
merged = spark.sql("""
SELECT
    p.place_name,
    p.place_type,
    ST_Union_Aggr(r.geom) AS merged_geom,
    COUNT(DISTINCT r.TRIP_ID) AS route_count
FROM routes_simplified r
JOIN 
    places p ON ST_Intersects(r.geom, p.geom)
GROUP BY p.place_name, p.place_type
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [26]:
merged.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+------------+--------------------+-----------+
|          place_name|  place_type|         merged_geom|route_count|
+--------------------+------------+--------------------+-----------+
|   allotments - 3581|  allotments|MULTILINESTRING (...|          1|
|   allotments - 7362|  allotments|MULTILINESTRING (...|          2|
|  brownfield - 14876|  brownfield|MULTILINESTRING (...|          2|
|  brownfield - 20453|  brownfield|MULTILINESTRING (...|          1|
|   brownfield - 3263|  brownfield|MULTILINESTRING (...|         55|
|     cemetery - 1661|    cemetery|MULTILINESTRING (...|          1|
|   commercial - 3955|  commercial|MULTILINESTRING (...|          3|
|   commercial - 4556|  commercial|MULTILINESTRING (...|          2|
|construction - 12825|construction|MULTILINESTRING (...|          2|
|construction - 20293|construction|LINESTRING (-8.68...|          1|
|construction - 20478|construction|MULTILINESTRING (...|          6|
|construction - 20507|construction

In [ ]:
#Decision: Where to position idle drivers 
#"Pre-position 15 cars near ???? and the city center during evening peak."
#---
#Where to invest in road infrastructure
#hottest cells / densest merged geometries reveal the most-traveled corridors

In [ ]:
#Many route LineStrings in a place        ST_Union_Aggr           One merged geometry
#┌──────────────────────┐
#│  ╱                   │
#│ ╱   ╲    ╱           │              ─────────▶           ┌──────────────────────┐
#│╱     ╲  ╱            │                                   │  ╱╲   ╱╲             │
#│       ╲╱             │                                   │ ╱  ╲ ╱  ╲   (all     │
#│   ╱╲                 │                                   │╱    ╳    ╲   routes  │
#│  ╱  ╲                │                                   │      ╲    ╲  combined│
#└──────────────────────┘                                   └──────────────────────┘
#  (3 separate routes)                                        (1 unified geometry)

In [ ]:
# ---- End Merge routes 

In [27]:
## SAVE GOLD LAYER - LAKE S3
#merged.limit(2).write.format("geoparquet").mode("overwrite").save("s3://workshop-sedona-datalake/gold/routes_merged.geoparquet/")
heatmap.write.format("parquet").mode("overwrite").save("s3://workshop-sedona-datalake/gold/heatmat.parquet/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [16]:
# ---- Scale demo: full pass-through aggregation, timed ----
df_gold_top_places = spark.sql("""
SELECT
    p.place_type,
    p.place_name,
    COUNT(DISTINCT r.TRIP_ID) AS routes,
    ROUND(AVG(r.length_meters), 0) AS avg_route_len_m
FROM routes_simplified r
JOIN 
    places p ON ST_Intersects(r.geom, p.geom)
GROUP BY p.place_type, p.place_name
ORDER BY routes DESC
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
df_gold_top_places.show(truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+--------------------+------+---------------+
|place_type  |place_name          |routes|avg_route_len_m|
+------------+--------------------+------+---------------+
|railway     |railway - 21759     |3786  |5221.0         |
|railway     |railway - 21761     |2090  |6367.0         |
|railway     |railway - 21766     |2060  |5129.0         |
|construction|construction - 21763|1798  |4940.0         |
|construction|construction - 21764|1429  |6154.0         |
|construction|construction - 21765|1213  |5574.0         |
|military    |military - 1217     |942   |6083.0         |
|construction|construction - 13276|873   |4458.0         |
|construction|construction - 13761|814   |5811.0         |
|industrial  |industrial - 68     |763   |11174.0        |
|grass       |grass - 11383       |722   |11977.0        |
|residential |residential - 8235  |713   |6312.0         |
|grass       |grass - 11382       |663   |11960.0        |
|grass       |grass - 11387       |648   |11545.0       

In [ ]:
## SAVE INTO GOLD LAYER - ATHENA

In [28]:
#Set your S3 bucket and Glue target
bucket = "workshop-sedona-warehouse"  
db_name = "`workshop-sedona`"
table_name = "gold_top_places"
table_path = f"s3://{bucket}/warehouse/{db_name}/{table_name}/"

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [29]:
df_gold_top_places.write.mode("overwrite").format("parquet").option("path", table_path).saveAsTable(f"{db_name}.{table_name}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…